# PS06-AI — Mẫu thực hành Bài 24–28
**Giảng viên: Nguyễn Bích Vân — IAI-UET-VNU**

Bản này có một ví dụ học cú pháp dùng số liệu riêng; không có lời giải điền sẵn cho bài được giao. Các mô phỏng đều là giả lập phục vụ học tập. **Không cần CSV đầu vào.**

Chọn kernel Python trong VS Code và chạy các ô từ trên xuống. Nếu thiếu thư viện, chạy trong một ô mới: `%pip install -r requirements.txt`, sau đó khởi động lại kernel. Kết quả hiện dưới ô mã, không phải Terminal.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats
from plotnine import ggplot, aes, geom_col, geom_point, labs
from IPython.display import display

OUTPUT_DIR = Path.cwd() / "student_outputs"
print("Python:", sys.executable)
print("Thu muc ket qua:", OUTPUT_DIR.resolve())

## 1. Ví dụ mẫu: ba phép kiểm tra, mỗi phép đạt với xác suất 0,7
Ví dụ này **khác số liệu Bài 11/24**, chỉ để học cách xây PMF và đối chiếu mô phỏng.

`pmf(k)` tính xác suất tại k; `cdf(k)` tính xác suất không quá k; `sf(k)` tính xác suất lớn hơn k. Với biến nguyên, “ít nhất k” dùng `sf(k-1)`.

Hàm dưới đây kiểm tra tham số, giữ cả giá trị có tần số 0 và không chuẩn hóa lại PMF sau khi cắt phần đuôi của Poisson.

In [ ]:
def build_model(family, params):
    """Tạo phân phối; tên tham số theo SciPy, không theo ký hiệu trên giấy."""
    if family == "binomial":
        n, p = params["n"], params["p"]
        if int(n) != n or n < 0 or not 0 <= p <= 1:
            raise ValueError("Can n nguyen khong am va 0 <= p <= 1.")
        return stats.binom(n=int(n), p=p)
    if family == "hypergeometric":
        M, K, s = params["M"], params["n"], params["N"]
        if any(int(v) != v for v in [M, K, s]) or M < 1 or not (0 <= K <= M and 0 <= s <= M):
            raise ValueError("Kiem tra M=tong the, n=loai can dem, N=co mau.")
        return stats.hypergeom(M=int(M), n=int(K), N=int(s))
    if family == "poisson":
        mu = params["mu"]
        if mu < 0:
            raise ValueError("mu phai khong am.")
        return stats.poisson(mu=mu)
    raise ValueError("family: binomial, hypergeometric hoac poisson.")


def compare_distribution(config, filename):
    """Chạy khi đã điền cấu hình; không tạo bảng rỗng khi chưa hoàn thành."""
    if config is None:
        print("Chua dien CONFIG. Doc de va tu dien tham so truoc khi chay.")
        return None
    B = int(config["n_simulations"])
    if B <= 0:
        raise ValueError("n_simulations phai duong.")
    dist = build_model(config["family"], config["params"])
    rng = np.random.default_rng(config["seed"])
    sample = dist.rvs(size=B, random_state=rng)
    lo, hi = dist.support()
    hi = int(hi) if np.isfinite(hi) else max(int(dist.ppf(0.999999)), int(sample.max()))
    support = np.arange(int(lo), hi + 1)
    counts = pd.Series(sample).value_counts().reindex(support, fill_value=0)
    table = pd.DataFrame({"value": support, "count": counts.to_numpy(),
                          "frequency": counts.to_numpy()/B,
                          "probability": dist.pmf(support)})
    table["absolute_error"] = abs(table["frequency"]-table["probability"])
    assert table["count"].sum() == B
    print("Phan xac suat ly thuyet ngoai mien ve:", float(dist.sf(hi)))
    display(table)
    plot = (ggplot(table, aes(x="value", y="frequency"))
            + geom_col() + geom_point(aes(y="probability"), size=3)
            + labs(x="Gia tri", y="Tan suat / xac suat", title="Cham: ly thuyet"))
    display(plot)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_DIR / f"{filename}.csv", index=False)
    plot.save(str(OUTPUT_DIR / f"{filename}.png"), width=7, height=4, dpi=150, verbose=False)
    return sample, table

In [ ]:
CONFIG = {"family": "binomial", "params": {"n": 3, "p": 0.7},
          "n_simulations": 4000, "seed": 2026}
example_sample, example_table = compare_distribution(CONFIG, "learning_example")

## 2. Cách chuyển từ mã mẫu sang bài được giao

Không chép số liệu ví dụ mẫu sang bài nộp. Trước khi thay `CONFIG`, ghi biến ngẫu nhiên, tên phân phối, tham số và biến cố theo đề. Tên cột trong bảng: `value` = giá trị, `count` = số lần, `frequency` = tần suất, `probability` = xác suất lý thuyết, `absolute_error` = sai lệch tuyệt đối.

Các ô `CONFIGxx = None` là vị trí cần tự điền. Khi chưa điền, chương trình chỉ nhắc việc cần làm; không phát sinh lỗi và không ghi CSV rỗng.

## Bài 24 — Dựng PMF lý thuyết
Đọc Bài 11 để tự điền số phép kiểm tra và xác suất đạt. Bài này yêu cầu bảng PMF và hình lý thuyết, không chỉ hình mô phỏng. Dùng np.arange và stats.binom.pmf, rồi geom_col.

**Lập luận trước khi chạy:** [Sinh viên điền tại đây.]

In [ ]:
CONFIG24 = None  # TODO: tu dien theo de, khong dung so lieu vi du mau.
result24 = compare_distribution(CONFIG24, "exercise24")

**Kết luận sau khi chạy:** [Ghi công thức, nhận xét sai lệch và giải thích. Không chỉ dán bảng.]

## Bài 25 — So sánh 500 và 10 000 lô
Sinh một mảng có 10 000 hàng, mỗi hàng là một lô năm quyết định. Dùng tổng theo hàng; lấy 500 và 10 000 hàng đầu để so sánh. Không gọi hai mô phỏng độc lập rồi mô tả chúng là hai tiền tố.

**Lập luận trước khi chạy:** [Sinh viên điền tại đây.]

In [ ]:
CONFIG25 = None  # TODO: tu dien theo de, khong dung so lieu vi du mau.
result25 = compare_distribution(CONFIG25, "exercise25")

**Kết luận sau khi chạy:** [Ghi công thức, nhận xét sai lệch và giải thích. Không chỉ dán bảng.]

## Bài 26 — Định tuyến 80 yêu cầu
Tự xác định n, p và ngưỡng “ít nhất”. Sau khi có sample, tạo mảng logic, lấy sum để đếm và mean để tính tần suất. Dùng sf với đúng đầu mút.

**Lập luận trước khi chạy:** [Sinh viên điền tại đây.]

In [ ]:
CONFIG26 = None  # TODO: tu dien theo de, khong dung so lieu vi du mau.
result26 = compare_distribution(CONFIG26, "exercise26")

**Kết luận sau khi chạy:** [Ghi công thức, nhận xét sai lệch và giải thích. Không chỉ dán bảng.]

## Bài 27 — Chọn đúng loại mẫu cần đếm
Đọc lại ý nghĩa M, n, N trong đề. Hai tham số n và N không có ý nghĩa giống nhau. Kiểm tra tên biến đang đếm trước khi mô phỏng.

**Lập luận trước khi chạy:** [Sinh viên điền tại đây.]

In [ ]:
CONFIG27 = None  # TODO: tu dien theo de, khong dung so lieu vi du mau.
result27 = compare_distribution(CONFIG27, "exercise27")

**Kết luận sau khi chạy:** [Ghi công thức, nhận xét sai lệch và giải thích. Không chỉ dán bảng.]

## Bài 28 — Ba mô hình
Tạo ba cấu hình riêng. Tự chọn biến cố cho từng cấu hình; bảng phải ghi rõ tham số, biến cố, xác suất lý thuyết, số lần, tần suất và sai lệch.

**Lập luận trước khi chạy:** [Sinh viên điền tại đây.]

In [ ]:
CONFIG28 = None  # TODO: tu dien theo de, khong dung so lieu vi du mau.
result28 = compare_distribution(CONFIG28, "exercise28")

**Kết luận sau khi chạy:** [Ghi công thức, nhận xét sai lệch và giải thích. Không chỉ dán bảng.]

## Tự kiểm tra trước khi nộp
Tổng tần số phải bằng số mô phỏng, mỗi tần suất nằm trong [0,1]. Nếu vẽ toàn bộ miền hữu hạn, tổng PMF phải xấp xỉ 1. Với Poisson cắt miền vẽ, phải báo phần xác suất còn lại, không gán nó bằng 0.

Đổi seed không đổi xác suất lý thuyết. Không có bảo đảm sai lệch giảm ở mọi giá trị sau mỗi lần tăng số mô phỏng. Xem `PS06_AI_Practice_Guide.md` để đối chiếu tham số và lỗi thường gặp.